<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/GPT_5DOT6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://platform.openai.com/home

https://developers.openai.com/api/docs

In [27]:
from google.colab  import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [4]:
# ============================================================================
# CHECK AVAILABLE OPENAI MODELS
# ============================================================================
import os
from openai import OpenAI

OPENAI_API_KEY = os.environ['OPENAI_API_KEY']
client = OpenAI(api_key=OPENAI_API_KEY)

import time
from datetime import datetime
import pytz

# Set the time zone to EST
est = pytz.timezone('EST')

# Current timestamp
start_time = time.time()
print(f"Timestamp: {start_time}")

# Current readable date and time in EST
est_time = datetime.now(est).strftime("%Y-%m-%d %H:%M:%S %Z")
print(f"Current Time (EST): {est_time}")

print("="*70)
print("CHECKING AVAILABLE OPENAI MODELS")
print("="*70)

try:
    models = client.models.list()

    # Filter for GPT models
    gpt_models = [m.id for m in models.data if 'gpt' in m.id.lower()]

    print(f"\n✅ Total models available: {len(models.data)}")
    print(f"✅ GPT models found: {len(gpt_models)}")
    print("\nAvailable GPT models:")
    print("-"*50)

    # Sort and display all GPT models
    for m in sorted(gpt_models):
        print(f"   {m}")

    # Check specifically for GPT-5.6
    gpt56_models = [m for m in gpt_models if '5.6' in m or '5-6' in m or '56' in m]
    if gpt56_models:
        print(f"\n✅ GPT-5.6 models found: {gpt56_models}")
    else:
        print(f"\n❌ No GPT-5.6 models found in available list")

    # Check for o1 (reasoning models)
    o1_models = [m for m in gpt_models if 'o1' in m or 'o-1' in m]
    if o1_models:
        print(f"\n✅ o1 reasoning models found: {o1_models}")

    # Check for latest models
    latest_models = [m for m in gpt_models if '4o' in m or '4-turbo' in m or '4-0125' in m]
    if latest_models:
        print(f"\n✅ Latest GPT-4 models found: {latest_models}")

    print("\n" + "="*70)
    print("RECOMMENDED MODEL FOR EXPERIMENT")
    print("="*70)

    # Priority order: GPT-5.6 (if exists) > o1 > GPT-4o > GPT-4-turbo
    priority = [
        *[m for m in gpt_models if '5.6' in m],
        *[m for m in gpt_models if 'o1' in m],
        *[m for m in gpt_models if '4o' in m],
        *[m for m in gpt_models if '4-turbo' in m],
        *[m for m in gpt_models if '4' in m and 'turbo' not in m],
        *gpt_models
    ]

    # Remove duplicates while preserving order
    seen = set()
    unique_priority = []
    for m in priority:
        if m not in seen:
            seen.add(m)
            unique_priority.append(m)

    if unique_priority:
        best_model = unique_priority[0]
        print(f"\n🏆 Best available model: {best_model}")
        print(f"   (This is the most capable model available to you)")

        # Show all options in priority order
        print(f"\n📋 All available models in priority order:")
        for i, m in enumerate(unique_priority[:10], 1):
            print(f"   {i}. {m}")
        if len(unique_priority) > 10:
            print(f"   ... and {len(unique_priority)-10} more")
    else:
        print("\n⚠️ No GPT models found!")
        best_model = "gpt-4o"  # fallback

except Exception as e:
    print(f"\n❌ Error checking models: {e}")
    print("Using fallback: gpt-4o")
    best_model = "gpt-4o"

print("\n" + "="*70)

Timestamp: 1783631434.756398
Current Time (EST): 2026-07-09 16:10:34 EST
CHECKING AVAILABLE OPENAI MODELS

✅ Total models available: 141
✅ GPT models found: 108

Available GPT models:
--------------------------------------------------
   chatgpt-image-latest
   ft:gpt-3.5-turbo-0125:xamrysoft::9cSMOUWb
   ft:gpt-3.5-turbo-0125:xamrysoft::BFmGnb0y:ckpt-step-72
   ft:gpt-3.5-turbo-0125:xamrysoft::BFmGqG94:ckpt-step-84
   ft:gpt-3.5-turbo-0125:xamrysoft::BFmGttyM
   ft:gpt-3.5-turbo-0125:xamrysoft::BFqZ0t1n:ckpt-step-534
   ft:gpt-3.5-turbo-0125:xamrysoft::BFqZ4Cdn:ckpt-step-1068
   ft:gpt-3.5-turbo-0125:xamrysoft::BFqZ9l56
   ft:gpt-3.5-turbo-0125:xamrysoft::BFrv8JcD:ckpt-step-1068
   ft:gpt-3.5-turbo-0125:xamrysoft::BFrv8bF5
   ft:gpt-3.5-turbo-0125:xamrysoft::BFrv8lmq:ckpt-step-534
   gpt-3.5-turbo
   gpt-3.5-turbo-0125
   gpt-3.5-turbo-1106
   gpt-3.5-turbo-16k
   gpt-3.5-turbo-instruct
   gpt-3.5-turbo-instruct-0914
   gpt-4
   gpt-4-0613
   gpt-4-turbo
   gpt-4-turbo-2024-04-09
   g

In [17]:
# ============================================================================
# CELL 1: PROVE GPT-5.6 REACHABILITY - CORRECTED SCHEMA
# ============================================================================
import os
import json
from datetime import datetime
from openai import OpenAI

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY', ''))

# List of models confirmed in your registry
GPT56_MODELS = [
    "gpt-5.6-sol",
    "gpt-5.6-terra",
    "gpt-5.6-luna"
]

results = []

print(f"\n🔍 TESTING GPT-5.6 COMPLETION SCHEMA")
print("-" * 75)

for model in GPT56_MODELS:
    try:
        # NOTE: Using max_completion_tokens as required by the new model schema
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Say hello"}],
            max_completion_tokens=5
        )
        results.append({"model": model, "status": "✅ SUCCESS", "error": None})
        print(f"✅ {model} → SUCCESS: {response.choices[0].message.content.strip()}")
    except Exception as e:
        results.append({"model": model, "status": "❌ ERROR", "error": str(e)[:200]})
        print(f"❌ {model} → ERROR: {str(e)[:80]}...")

# Save updated evidence
with open("gpt56_status_evidence.json", "w") as f:
    json.dump({"test_date": datetime.now().isoformat(), "results": results}, f, indent=2)

print(f"\n💾 Updated evidence saved to: gpt56_status_evidence.json")


🔍 TESTING GPT-5.6 COMPLETION SCHEMA
---------------------------------------------------------------------------
✅ gpt-5.6-sol → SUCCESS: 
✅ gpt-5.6-terra → SUCCESS: 
✅ gpt-5.6-luna → SUCCESS: 

💾 Updated evidence saved to: gpt56_status_evidence.json


In [29]:
#!/usr/bin/env python3
"""
GPT-5.6 AGENTIC VALIDATOR - COLAB OPTIMIZED
Properly loads API key from Colab userdata
"""

# ============================================================================
# CELL 1: SETUP & API KEY
# ============================================================================

from google.colab import userdata
import os

# Load API key from Colab secrets
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

print(f"✅ API Key loaded: {'*' * 10}{os.environ['OPENAI_API_KEY'][-4:]}")

# ============================================================================
# CELL 2: IMPORTS
# ============================================================================

import json
import time
import asyncio
import logging
import statistics
from datetime import datetime
from typing import Dict, List, Optional, Any
from dataclasses import dataclass, asdict, field
from enum import Enum

from openai import AsyncOpenAI, OpenAI
import numpy as np

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports loaded")

# ============================================================================
# CELL 3: CONSTANTS & DATA CLASSES
# ============================================================================

class ModelTier(Enum):
    SOL = "gpt-5.6-sol"
    TERRA = "gpt-5.6-terra"
    LUNA = "gpt-5.6-luna"

@dataclass
class TestResult:
    """Individual test result"""
    model: str
    test_name: str
    status: str  # "SUCCESS", "WARNING", "ERROR"
    latency_ms: float
    tokens_used: int
    completion_tokens: int
    prompt_tokens: int
    response_preview: str
    timestamp: str
    error: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class ModelHealth:
    """Health status for a model"""
    model: str
    success_rate: float
    avg_latency_ms: float
    p95_latency_ms: float
    total_requests: int
    error_count: int
    avg_tokens_per_request: float
    health_score: float
    status: str  # "HEALTHY", "DEGRADED", "UNHEALTHY"
    last_updated: str

@dataclass
class AgentState:
    """Persistent agent state"""
    version: str = "1.0.0"
    last_run: Optional[str] = None
    total_tests: int = 0
    model_health: Dict[str, ModelHealth] = field(default_factory=dict)
    test_history: List[TestResult] = field(default_factory=list)
    alert_history: List[Dict] = field(default_factory=dict)

print("✅ Data classes defined")

# ============================================================================
# CELL 4: CORE AGENTIC ENGINE
# ============================================================================

class GPT56AgenticValidator:
    """Intelligent validation agent for GPT-5.6 models - Colab Optimized"""

    def __init__(self):
        """Initialize validator with API key from environment"""
        self.api_key = os.environ.get('OPENAI_API_KEY', '')
        if not self.api_key:
            raise ValueError("OPENAI_API_KEY not found in environment!")

        # Initialize clients
        self.client = AsyncOpenAI(api_key=self.api_key)
        self.sync_client = OpenAI(api_key=self.api_key)

        # Models to test
        self.models = [t.value for t in ModelTier]

        # State
        self.state = AgentState()

        # Thresholds
        self.thresholds = {
            'gpt-5.6-sol': {'max_latency_ms': 3000, 'min_success_rate': 0.95},
            'gpt-5.6-terra': {'max_latency_ms': 2000, 'min_success_rate': 0.95},
            'gpt-5.6-luna': {'max_latency_ms': 1000, 'min_success_rate': 0.95}
        }

        # Test templates
        self.test_templates = self._init_test_templates()

        print(f"🧠 GPT-5.6 Agentic Validator initialized")
        print(f"📋 Models: {', '.join(self.models)}")
        print(f"🔑 API Key: {'*' * 10}{self.api_key[-4:]}")
        print(f"📊 State: {len(self.state.test_history)} previous tests")

    def _init_test_templates(self) -> Dict[str, List[Dict]]:
        """Initialize test templates by category"""
        return {
            "connectivity": [
                {"name": "basic_ping", "prompt": "Say 'Hello' and confirm your model name."},
                {"name": "simple_math", "prompt": "What is 15 * 37 + 28? Answer briefly."}
            ],
            "reasoning": [
                {"name": "logical", "prompt": "If all A are B, and all B are C, what can you conclude? Answer briefly."},
                {"name": "code_gen", "prompt": "Write a Python function to calculate factorial recursively."}
            ],
            "performance": [
                {"name": "short_response", "prompt": "Explain quantum computing in 30 words.", "max_tokens": 50},
                {"name": "long_response", "prompt": "Describe neural networks in 100 words.", "max_tokens": 150}
            ],
            "agentic": [
                {"name": "planning", "prompt": "Plan a 3-day trip to Paris with a $2000 budget. Provide 5 bullet points."},
                {"name": "analysis", "prompt": "Analyze the pros and cons of remote work. Provide 3 pros and 3 cons."}
            ]
        }

    # ========================================================================
    # TEST EXECUTION
    # ========================================================================

    async def run_test(self, model: str, test: Dict, category: str) -> TestResult:
        """Run a single test"""
        start = time.time()
        test_name = test.get('name', 'unknown')

        try:
            response = await asyncio.wait_for(
                self.client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": "You are a helpful assistant. Provide clear, accurate, and concise responses."},
                        {"role": "user", "content": test.get('prompt', 'Hello')}
                    ],
                    max_completion_tokens=test.get('max_tokens', 100),
                    temperature=0.3
                ),
                timeout=30.0
            )

            latency = (time.time() - start) * 1000
            response_text = response.choices[0].message.content

            result = TestResult(
                model=model,
                test_name=f"{category}/{test_name}",
                status="SUCCESS",
                latency_ms=latency,
                tokens_used=response.usage.total_tokens,
                completion_tokens=response.usage.completion_tokens,
                prompt_tokens=response.usage.prompt_tokens,
                response_preview=response_text[:200],
                timestamp=datetime.now().isoformat()
            )

            logger.info(f"✅ {model} - {category}/{test_name}: {latency:.0f}ms")

        except asyncio.TimeoutError:
            latency = (time.time() - start) * 1000
            result = TestResult(
                model=model,
                test_name=f"{category}/{test_name}",
                status="ERROR",
                latency_ms=latency,
                tokens_used=0,
                completion_tokens=0,
                prompt_tokens=0,
                response_preview="",
                timestamp=datetime.now().isoformat(),
                error="Timeout after 30 seconds"
            )
            logger.error(f"⏰ {model} - {category}/{test_name}: TIMEOUT")

        except Exception as e:
            latency = (time.time() - start) * 1000
            error_msg = str(e)

            # Clean up error message
            if "400" in error_msg:
                error_msg = "Bad Request - Check model name and parameters"
            elif "401" in error_msg:
                error_msg = "Authentication Error - Check API key"
            elif "429" in error_msg:
                error_msg = "Rate Limit Exceeded - Slow down requests"
            elif "insufficient_quota" in error_msg:
                error_msg = "Insufficient Quota - Check your OpenAI billing"

            result = TestResult(
                model=model,
                test_name=f"{category}/{test_name}",
                status="ERROR",
                latency_ms=latency,
                tokens_used=0,
                completion_tokens=0,
                prompt_tokens=0,
                response_preview="",
                timestamp=datetime.now().isoformat(),
                error=error_msg[:200]
            )
            logger.error(f"❌ {model} - {category}/{test_name}: {error_msg[:80]}")

        return result

    async def run_test_suite(self, iterations: int = 2) -> Dict[str, List[TestResult]]:
        """Run complete test suite"""

        print(f"\n🚀 RUNNING AGENTIC TEST SUITE")
        print("=" * 80)
        print(f"Models: {', '.join(self.models)}")
        print(f"Categories: {', '.join(self.test_templates.keys())}")
        print(f"Iterations per test: {iterations}")
        print("=" * 80)

        all_results = {}

        for model in self.models:
            print(f"\n🎯 TESTING {model.upper()}")
            print("-" * 60)

            model_results = []

            for category, tests in self.test_templates.items():
                for test in tests:
                    for i in range(iterations):
                        # Add variation for multiple iterations
                        if i > 0:
                            test_copy = test.copy()
                            test_copy['prompt'] = f"{test['prompt']} (variant {i+1})"
                        else:
                            test_copy = test

                        result = await self.run_test(model, test_copy, category)
                        model_results.append(result)

                        # Print progress with emoji
                        emoji = "✅" if result.status == "SUCCESS" else "❌"
                        error_info = f" - {result.error[:50]}" if result.error else ""
                        print(f"  {emoji} {result.test_name}: {result.status} ({result.latency_ms:.0f}ms){error_info}")

                        # Rate limiting
                        await asyncio.sleep(0.5)

            all_results[model] = model_results
            self.state.test_history.extend(model_results)

            # Update health
            health = self._update_health(model, model_results)
            print(f"\n📊 {model} Health: {health.health_score:.1f}% ({health.status})")

        # Update state
        self.state.last_run = datetime.now().isoformat()
        self.state.total_tests += sum(len(r) for r in all_results.values())

        return all_results

    # ========================================================================
    # HEALTH MANAGEMENT
    # ========================================================================

    def _update_health(self, model: str, results: List[TestResult]) -> ModelHealth:
        """Update model health metrics"""

        if not results:
            return ModelHealth(
                model=model,
                success_rate=0.0,
                avg_latency_ms=0.0,
                p95_latency_ms=0.0,
                total_requests=0,
                error_count=0,
                avg_tokens_per_request=0.0,
                health_score=0.0,
                status="UNKNOWN",
                last_updated=datetime.now().isoformat()
            )

        total = len(results)
        successes = sum(1 for r in results if r.status == "SUCCESS")
        errors = sum(1 for r in results if r.status == "ERROR")
        warnings = sum(1 for r in results if r.status == "WARNING")

        success_rate = successes / total if total > 0 else 0
        latencies = [r.latency_ms for r in results if r.latency_ms > 0]
        tokens = [r.tokens_used for r in results if r.tokens_used > 0]

        avg_latency = statistics.mean(latencies) if latencies else 0
        p95_latency = np.percentile(latencies, 95) if latencies else 0
        avg_tokens = statistics.mean(tokens) if tokens else 0

        # Health score: 60% success, 30% latency, 10% stability
        latency_score = max(0, min(100, 100 - (avg_latency / 50)))
        success_score = success_rate * 100
        stability_score = 100 * (1 - (errors / total)) if total > 0 else 0

        health_score = (success_score * 0.6) + (latency_score * 0.3) + (stability_score * 0.1)
        health_score = min(100, max(0, health_score))

        if health_score >= 80:
            status = "HEALTHY"
        elif health_score >= 50:
            status = "DEGRADED"
        else:
            status = "UNHEALTHY"

        health = ModelHealth(
            model=model,
            success_rate=success_rate,
            avg_latency_ms=avg_latency,
            p95_latency_ms=p95_latency,
            total_requests=total,
            error_count=errors,
            avg_tokens_per_request=avg_tokens,
            health_score=health_score,
            status=status,
            last_updated=datetime.now().isoformat()
        )

        self.state.model_health[model] = health

        # Generate alerts if needed
        self._check_thresholds(model, health)

        return health

    def _check_thresholds(self, model: str, health: ModelHealth):
        """Check health thresholds and generate alerts"""

        model_key = model.lower()
        thresholds = self.thresholds.get(model_key, self.thresholds.get('gpt-5.6-sol'))

        # Success rate alert
        if health.success_rate < thresholds.get('min_success_rate', 0.95):
            alert = {
                "timestamp": datetime.now().isoformat(),
                "model": model,
                "severity": "critical",
                "message": f"Success rate dropped to {health.success_rate:.1%} (threshold: {thresholds['min_success_rate']:.1%})"
            }
            self.state.alert_history.append(alert)
            logger.critical(f"🚨 CRITICAL: {model} - {alert['message']}")

        # Latency alert
        if health.avg_latency_ms > thresholds.get('max_latency_ms', 3000):
            alert = {
                "timestamp": datetime.now().isoformat(),
                "model": model,
                "severity": "warning",
                "message": f"Average latency increased to {health.avg_latency_ms:.0f}ms"
            }
            self.state.alert_history.append(alert)
            logger.warning(f"⚠️ WARNING: {model} - {alert['message']}")

        # Health score alert
        if health.health_score < 50:
            alert = {
                "timestamp": datetime.now().isoformat(),
                "model": model,
                "severity": "critical",
                "message": f"Health score dropped to {health.health_score:.1f}%"
            }
            self.state.alert_history.append(alert)
            logger.critical(f"🚨 CRITICAL: {model} - {alert['message']}")

    # ========================================================================
    # REPORTING
    # ========================================================================

    def print_summary(self, results: Dict[str, List[TestResult]]):
        """Print comprehensive test summary"""

        print(f"\n" + "=" * 80)
        print("📊 TEST SUMMARY")
        print("=" * 80)

        total_tests = 0
        total_success = 0
        total_error = 0

        for model, model_results in results.items():
            success = sum(1 for r in model_results if r.status == "SUCCESS")
            error = sum(1 for r in model_results if r.status == "ERROR")
            warning = sum(1 for r in model_results if r.status == "WARNING")
            total = len(model_results)

            total_tests += total
            total_success += success
            total_error += error

            health = self.state.model_health.get(model)

            print(f"\n{model.upper()}:")
            print(f"  ✅ Success: {success}/{total} ({success/total*100:.1f}%)")
            print(f"  ⚠️ Warnings: {warning}/{total} ({warning/total*100:.1f}%)")
            print(f"  ❌ Errors: {error}/{total} ({error/total*100:.1f}%)")
            if health:
                print(f"  💚 Health: {health.status} ({health.health_score:.1f}%)")
                print(f"  ⏱️ Avg Latency: {health.avg_latency_ms:.0f}ms")
                print(f"  📝 Avg Tokens: {health.avg_tokens_per_request:.0f}")

        overall_success_rate = (total_success / total_tests * 100) if total_tests > 0 else 0

        print(f"\n" + "-" * 80)
        print(f"📈 OVERALL STATISTICS:")
        print(f"  Total Tests: {total_tests}")
        print(f"  Success Rate: {overall_success_rate:.1f}%")
        print(f"  Total Errors: {total_error}")
        print(f"  Total Time: {sum(r.latency_ms for r in self.state.test_history) / 1000:.1f}s")

        # Print alerts
        if self.state.alert_history:
            print(f"\n🚨 ALERTS: {len(self.state.alert_history)}")
            for alert in self.state.alert_history[-5:]:
                print(f"  • {alert['severity'].upper()}: {alert['message']}")

        # Save state
        with open("gpt56_agent_state.json", "w") as f:
            json.dump(asdict(self.state), f, indent=2)
        print(f"\n💾 State saved to: gpt56_agent_state.json")

    # ========================================================================
    # MAIN EXECUTION
    # ========================================================================

    async def run_full_validation(self, iterations: int = 2) -> Dict[str, Any]:
        """Run complete validation workflow"""

        print("\n" + "=" * 80)
        print("🧠 GPT-5.6 AGENTIC VALIDATION SYSTEM")
        print("=" * 80)
        print(f"Time: {datetime.now().isoformat()}")
        print(f"Iterations per test: {iterations}")

        start_time = time.time()

        # Run tests
        results = await self.run_test_suite(iterations=iterations)

        elapsed = time.time() - start_time

        # Print summary
        self.print_summary(results)

        print(f"\n⏱️ Total execution time: {elapsed:.1f}s")

        return {
            "status": "completed",
            "timestamp": datetime.now().isoformat(),
            "total_tests": self.state.total_tests,
            "models_tested": list(results.keys()),
            "execution_time_seconds": elapsed,
            "state_file": "gpt56_agent_state.json",
            "alerts": self.state.alert_history
        }

# ============================================================================
# CELL 5: MAIN EXECUTION
# ============================================================================

async def main():
    """Main entry point"""
    try:
        validator = GPT56AgenticValidator()
        result = await validator.run_full_validation(iterations=2)

        print("\n" + "=" * 80)
        print("✅ VALIDATION COMPLETE")
        print("=" * 80)
        print(f"Status: {result['status']}")
        print(f"Total Tests: {result['total_tests']}")
        print(f"Models Tested: {', '.join(result['models_tested'])}")
        print(f"Execution Time: {result['execution_time_seconds']:.1f}s")
        print(f"State File: {result['state_file']}")
        if result.get('alerts'):
            print(f"Alerts: {len(result['alerts'])}")
        print("=" * 80)

        return result

    except Exception as e:
        print(f"\n❌ ERROR: {e}")
        import traceback
        traceback.print_exc()
        raise

# ============================================================================
# CELL 6: RUN IN COLAB
# ============================================================================

# Check if running in Colab
try:
    from IPython import get_ipython
    if get_ipython():
        print("📓 Running in Google Colab")
        print("💡 Use: result = await main()")
    else:
        print("💻 Running as Python script")
        result = asyncio.run(main())
except ImportError:
    print("💻 Running as Python script")
    result = asyncio.run(main())

✅ API Key loaded: **********8UMA
✅ Imports loaded
✅ Data classes defined
📓 Running in Google Colab
💡 Use: result = await main()


## 📊 PERFORMANCE COMPARISON

In [6]:
# ============================================================================
# GPT-5.6 VALIDATOR - USING RESPONSES API (CORRECT)
# ============================================================================

from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
import time
from datetime import datetime

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

print("=" * 80)
print("🚀 GPT-5.6 AGENTIC VALIDATOR - RESPONSES API")
print("=" * 80)

# Models to test
MODELS = ["gpt-5.6-sol", "gpt-5.6-terra", "gpt-5.6-luna"]

# Test prompts
TEST_PROMPTS = [
    "Say 'Hello, I am GPT-5.6'",
    "What is 15 * 37 + 28?",
    "If all A are B, and all B are C, what can you conclude?",
    "Write a one-sentence bedtime story about AI",
]

results = []

print(f"\n📋 Testing {len(MODELS)} models with {len(TEST_PROMPTS)} prompts")
print("=" * 80)

for model in MODELS:
    print(f"\n🎯 TESTING: {model}")
    print("-" * 60)

    model_results = []

    for prompt in TEST_PROMPTS:
        start = time.time()

        try:
            # ✅ CORRECT: Using Responses API
            response = client.responses.create(
                model=model,
                input=prompt,  # Note: 'input' not 'messages'
            )

            latency = (time.time() - start) * 1000

            # ✅ CORRECT: Access response using output_text
            output_text = response.output_text

            print(f"  ✅ {prompt[:30]}...")
            print(f"     Response: {output_text[:100]}")
            print(f"     Latency: {latency:.0f}ms")

            model_results.append({
                "prompt": prompt,
                "status": "SUCCESS",
                "response": output_text,
                "latency_ms": latency
            })

        except Exception as e:
            latency = (time.time() - start) * 1000
            error_msg = str(e)
            print(f"  ❌ {prompt[:30]}... - {error_msg[:80]}")

            model_results.append({
                "prompt": prompt,
                "status": "ERROR",
                "error": error_msg,
                "latency_ms": latency
            })

        time.sleep(0.3)  # Rate limiting

    results.append({
        "model": model,
        "results": model_results
    })

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("📊 TEST SUMMARY")
print("=" * 80)

for model_result in results:
    model = model_result["model"]
    model_results = model_result["results"]

    successes = [r for r in model_results if r["status"] == "SUCCESS"]
    errors = [r for r in model_results if r["status"] == "ERROR"]

    print(f"\n{model}:")
    print(f"  ✅ Success: {len(successes)}/{len(model_results)}")
    print(f"  ❌ Errors: {len(errors)}/{len(model_results)}")

    if successes:
        avg_latency = sum(r["latency_ms"] for r in successes) / len(successes)
        print(f"  ⏱️ Avg Latency: {avg_latency:.0f}ms")

        # Show sample response
        print(f"  📝 Sample: {successes[0]['response'][:100]}...")

print("\n" + "=" * 80)
print("✅ VALIDATION COMPLETE")
print("=" * 80)

🚀 GPT-5.6 AGENTIC VALIDATOR - RESPONSES API

📋 Testing 3 models with 4 prompts

🎯 TESTING: gpt-5.6-sol
------------------------------------------------------------
  ✅ Say 'Hello, I am GPT-5.6'...
     Response: Hello, I am GPT-5.6
     Latency: 2363ms
  ✅ What is 15 * 37 + 28?...
     Response: 583
     Latency: 1090ms
  ✅ If all A are B, and all B are ...
     Response: All A are C.
     Latency: 1240ms
  ✅ Write a one-sentence bedtime s...
     Response: Each night, a little AI gathered the world’s happiest dreams and wove them into a lullaby for the st
     Latency: 1648ms

🎯 TESTING: gpt-5.6-terra
------------------------------------------------------------
  ✅ Say 'Hello, I am GPT-5.6'...
     Response: Hello, I am GPT-5.6
     Latency: 707ms
  ✅ What is 15 * 37 + 28?...
     Response: 583
     Latency: 1012ms
  ✅ If all A are B, and all B are ...
     Response: All A are C.
     Latency: 686ms
  ✅ Write a one-sentence bedtime s...
     Response: Under a sky of softly blinking st

## 🚀 FULL AGENTIC VALIDATOR WITH RESPONSES API

In [7]:
# ============================================================================
# GPT-5.6 AGENTIC VALIDATOR - COMPLETE SOLUTION
# ============================================================================

from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
import json
import time
from datetime import datetime
from typing import Dict, List, Optional
from dataclasses import dataclass, asdict

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

# ============================================================================
# DATA CLASSES
# ============================================================================

@dataclass
class TestResult:
    model: str
    prompt: str
    status: str
    response: str
    latency_ms: float
    timestamp: str
    error: Optional[str] = None

@dataclass
class ModelHealth:
    model: str
    success_rate: float
    avg_latency_ms: float
    total_tests: int
    errors: int
    health_score: float
    status: str
    capabilities: Dict[str, float]  # Reasoning, Speed, Accuracy

# ============================================================================
# GPT-5.6 AGENTIC VALIDATOR
# ============================================================================

class GPT56AgenticValidator:
    """Full GPT-5.6 Agentic Validator using Responses API"""

    def __init__(self):
        self.models = ["gpt-5.6-sol", "gpt-5.6-terra", "gpt-5.6-luna"]
        self.client = client
        self.results = []

        # Comprehensive test suite
        self.test_suite = {
            "basic": [
                "Say 'Hello, I am GPT-5.6'",
                "What is 15 * 37 + 28?",
                "What is the capital of France?",
            ],
            "reasoning": [
                "If all A are B, and all B are C, what can you conclude?",
                "A train leaves Station A at 60 mph. Another leaves Station B at 80 mph. Stations are 300 miles apart. When do they meet?",
                "Explain the Monty Hall problem in simple terms.",
            ],
            "creative": [
                "Write a one-sentence bedtime story about AI",
                "Write a haiku about artificial intelligence",
                "Create a metaphor for machine learning",
            ],
            "planning": [
                "Plan a 3-day trip to Paris with a $2000 budget in 5 bullet points",
                "Provide a step-by-step plan for learning Python in 30 days",
                "Design a weekly meal plan for a vegetarian athlete",
            ],
            "analysis": [
                "Analyze the pros and cons of remote work in 3 points each",
                "Compare the advantages of electric vs hydrogen vehicles",
                "Analyze the impact of AI on job markets",
            ],
            "coding": [
                "Write a Python function to calculate factorial recursively",
                "Write a Python function to check if a number is prime",
                "Write a simple HTML page with a button that says 'Click Me'",
            ]
        }

        print("🧠 GPT-5.6 Agentic Validator Initialized")
        print(f"📋 Models: {self.models}")
        print(f"📝 Test Categories: {len(self.test_suite)}")
        print(f"📊 Total Tests: {sum(len(tests) for tests in self.test_suite.values())}")

    def run_test(self, model: str, prompt: str, category: str) -> TestResult:
        """Run a single test using Responses API"""
        start = time.time()

        try:
            response = self.client.responses.create(
                model=model,
                input=prompt,
            )

            latency = (time.time() - start) * 1000
            output_text = response.output_text

            return TestResult(
                model=model,
                prompt=prompt[:60],
                status="SUCCESS",
                response=output_text[:300],
                latency_ms=latency,
                timestamp=datetime.now().isoformat()
            )

        except Exception as e:
            latency = (time.time() - start) * 1000
            return TestResult(
                model=model,
                prompt=prompt[:60],
                status="ERROR",
                response="",
                latency_ms=latency,
                timestamp=datetime.now().isoformat(),
                error=str(e)[:150]
            )

    def run_full_suite(self) -> Dict:
        """Run complete test suite across all models"""

        print("\n" + "=" * 80)
        print("🚀 RUNNING GPT-5.6 AGENTIC TEST SUITE")
        print("=" * 80)

        all_results = {}

        for model in self.models:
            print(f"\n🎯 TESTING: {model}")
            print("-" * 60)

            model_results = []
            test_count = 0

            for category, prompts in self.test_suite.items():
                print(f"\n  📂 Category: {category.upper()}")

                for prompt in prompts:
                    test_count += 1
                    result = self.run_test(model, prompt, category)
                    model_results.append(result)

                    # Print progress
                    emoji = "✅" if result.status == "SUCCESS" else "❌"
                    error_info = f" - {result.error[:50]}" if result.error else ""
                    print(f"    {emoji} Test {test_count}: {result.status} ({result.latency_ms:.0f}ms){error_info}")

                    if result.status == "SUCCESS":
                        preview = result.response[:60] + "..." if len(result.response) > 60 else result.response
                        print(f"       📝 {preview}")

                    time.sleep(0.2)  # Rate limiting

            all_results[model] = model_results
            self.results.extend(model_results)

            # Calculate health
            health = self._calculate_health(model, model_results)
            print(f"\n📊 {model} Health: {health.health_score:.1f}% ({health.status})")
            print(f"   ⏱️ Avg Latency: {health.avg_latency_ms:.0f}ms")
            print(f"   ✅ Success Rate: {health.success_rate*100:.1f}%")

        return all_results

    def _calculate_health(self, model: str, results: List[TestResult]) -> ModelHealth:
        """Calculate comprehensive model health metrics"""
        total = len(results)
        successes = sum(1 for r in results if r.status == "SUCCESS")
        errors = sum(1 for r in results if r.status == "ERROR")

        success_rate = successes / total if total > 0 else 0
        latencies = [r.latency_ms for r in results if r.latency_ms > 0]
        avg_latency = sum(latencies) / len(latencies) if latencies else 0

        # Metrics
        latency_score = max(0, min(100, 100 - (avg_latency / 30)))
        success_score = success_rate * 100
        stability_score = 100 * (1 - (errors / total)) if total > 0 else 0

        health_score = (success_score * 0.6) + (latency_score * 0.3) + (stability_score * 0.1)

        if health_score >= 80:
            status = "HEALTHY"
        elif health_score >= 50:
            status = "DEGRADED"
        else:
            status = "UNHEALTHY"

        # Capabilities assessment
        capabilities = {
            "reasoning": min(100, health_score * 0.9 + 10),
            "speed": min(100, 100 - (avg_latency / 20)),
            "accuracy": success_rate * 100,
            "stability": stability_score
        }

        return ModelHealth(
            model=model,
            success_rate=success_rate,
            avg_latency_ms=avg_latency,
            total_tests=total,
            errors=errors,
            health_score=health_score,
            status=status,
            capabilities=capabilities
        )

    def print_summary(self, all_results: Dict):
        """Print comprehensive test summary"""

        print("\n" + "=" * 80)
        print("📊 TEST SUMMARY")
        print("=" * 80)

        total_tests = 0
        total_success = 0
        total_errors = 0

        # Model comparisons
        model_health = {}
        for model, results in all_results.items():
            health = self._calculate_health(model, results)
            model_health[model] = health

            successes = sum(1 for r in results if r.status == "SUCCESS")
            errors = sum(1 for r in results if r.status == "ERROR")
            total = len(results)

            total_tests += total
            total_success += successes
            total_errors += errors

            print(f"\n{model.upper()}:")
            print(f"  ✅ Success: {successes}/{total} ({successes/total*100:.1f}%)")
            print(f"  ❌ Errors: {errors}/{total} ({errors/total*100:.1f}%)")
            print(f"  💚 Health: {health.status} ({health.health_score:.1f}%)")
            print(f"  ⏱️ Avg Latency: {health.avg_latency_ms:.0f}ms")
            print(f"  🧠 Reasoning: {health.capabilities['reasoning']:.1f}%")
            print(f"  ⚡ Speed: {health.capabilities['speed']:.1f}%")
            print(f"  🎯 Accuracy: {health.capabilities['accuracy']:.1f}%")

        print(f"\n" + "-" * 80)
        print(f"📈 OVERALL: {total_success}/{total_tests} successful ({total_success/total_tests*100:.1f}%)")

        # Recommendations
        print("\n💡 AGENTIC RECOMMENDATIONS:")
        print("-" * 60)

        # Find best model for each category
        if model_health:
            best_overall = max(model_health.items(), key=lambda x: x[1].health_score)
            best_speed = min(model_health.items(), key=lambda x: x[1].avg_latency_ms)
            best_reasoning = max(model_health.items(), key=lambda x: x[1].capabilities['reasoning'])

            print(f"\n  🏆 Best Overall: {best_overall[0].upper()} ({best_overall[1].health_score:.1f}%)")
            print(f"  ⚡ Fastest: {best_speed[0].upper()} ({best_speed[1].avg_latency_ms:.0f}ms)")
            print(f"  🧠 Best Reasoning: {best_reasoning[0].upper()} ({best_reasoning[1].capabilities['reasoning']:.1f}%)")

            print("\n  🎯 Use Cases:")
            print(f"    • {best_overall[0].upper()}: General purpose, most capable")
            print(f"    • {best_speed[0].upper()}: Speed-critical applications")

            if "sol" in str(best_reasoning[0]):
                print(f"    • GPT-5.6-SOL: Complex reasoning, coding, analysis")
            if "terra" in str(best_speed[0]):
                print(f"    • GPT-5.6-TERRA: Balanced performance, cost-effective")
            if "luna" in str(best_speed[0]):
                print(f"    • GPT-5.6-LUNA: High-speed, simple tasks")

        # Save results
        with open("gpt56_full_results.json", "w") as f:
            json.dump({
                "timestamp": datetime.now().isoformat(),
                "total_tests": total_tests,
                "success_rate": total_success / total_tests if total_tests > 0 else 0,
                "model_health": {k: asdict(v) for k, v in model_health.items()},
                "results": [
                    {
                        "model": r.model,
                        "prompt": r.prompt,
                        "status": r.status,
                        "response": r.response[:300],
                        "latency_ms": r.latency_ms,
                        "error": r.error
                    }
                    for r in self.results
                ]
            }, f, indent=2)

        print(f"\n💾 Results saved to: gpt56_full_results.json")

    def run(self) -> Dict:
        """Run full validation"""

        print("\n" + "=" * 80)
        print("🧠 GPT-5.6 AGENTIC VALIDATION SYSTEM")
        print("=" * 80)
        print(f"Time: {datetime.now().isoformat()}")

        start_time = time.time()

        # Run tests
        results = self.run_full_suite()

        elapsed = time.time() - start_time

        # Print summary
        self.print_summary(results)

        print(f"\n⏱️ Total execution time: {elapsed:.1f}s")
        print("=" * 80)
        print("✅ VALIDATION COMPLETE")
        print("=" * 80)

        return {
            "status": "completed",
            "timestamp": datetime.now().isoformat(),
            "execution_time": elapsed,
            "models_tested": len(self.models),
            "total_tests": sum(len(r) for r in results.values()),
            "results": results
        }

# ============================================================================
# RUN THE VALIDATOR
# ============================================================================

validator = GPT56AgenticValidator()
result = validator.run()

# Display final summary
print("\n" + "=" * 80)
print("🚀 GPT-5.6 DEPLOYMENT READY!")
print("=" * 80)
print(f"✅ All {result['models_tested']} models tested successfully")
print(f"📊 {result['total_tests']} total tests completed")
print(f"⏱️ Execution time: {result['execution_time']:.1f}s")
print("\n📋 Best Model Recommendations:")
print("  • General Purpose: gpt-5.6-sol")
print("  • Balanced Speed: gpt-5.6-terra")
print("  • Fast Responses: gpt-5.6-luna")
print("=" * 80)

🧠 GPT-5.6 Agentic Validator Initialized
📋 Models: ['gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-5.6-luna']
📝 Test Categories: 6
📊 Total Tests: 18

🧠 GPT-5.6 AGENTIC VALIDATION SYSTEM
Time: 2026-07-09T21:19:45.318442

🚀 RUNNING GPT-5.6 AGENTIC TEST SUITE

🎯 TESTING: gpt-5.6-sol
------------------------------------------------------------

  📂 Category: BASIC
    ✅ Test 1: SUCCESS (1087ms)
       📝 Hello, I am GPT-5.6
    ✅ Test 2: SUCCESS (1058ms)
       📝 583
    ✅ Test 3: SUCCESS (841ms)
       📝 Paris.

  📂 Category: REASONING
    ✅ Test 4: SUCCESS (845ms)
       📝 All A are C.
    ✅ Test 5: SUCCESS (2123ms)
       📝 Assuming they leave at the same time and travel toward each ...
    ✅ Test 6: SUCCESS (4830ms)
       📝 The **Monty Hall problem** is a probability puzzle:

1. You ...

  📂 Category: CREATIVE
    ✅ Test 7: SUCCESS (1851ms)
       📝 Each night, the little AI gathered the world’s kindest dream...
    ✅ Test 8: SUCCESS (2248ms)
       📝 Silent circuits dream  
Learning from the wor

## 🚀 AOCC AGENTIC SYSTEM WITH GPT-5.6

In [9]:
# ============================================================================
# AOCC AGENTIC SYSTEM WITH GPT-5.6 - FIXED
# ============================================================================

from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
import json
import time
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any
from dataclasses import dataclass, asdict, field
from enum import Enum
import random

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

# ============================================================================
# DATA CLASSES
# ============================================================================

class Priority(Enum):
    CRITICAL = 1
    HIGH = 2
    MEDIUM = 3
    LOW = 4

class IncidentType(Enum):
    FLIGHT_DELAY = "flight_delay"
    CREW_UNAVAILABLE = "crew_unavailable"
    WEATHER = "weather"
    MAINTENANCE = "maintenance"
    PASSENGER_ISSUE = "passenger_issue"
    REGULATORY = "regulatory"
    ATC = "atc"
    FUEL = "fuel"

@dataclass
class Flight:
    flight_number: str
    departure: str
    arrival: str
    departure_time: str
    arrival_time: str
    status: str
    gate: str
    aircraft: str
    passengers: int
    crew: List[str]

@dataclass
class Incident:
    id: str
    type: IncidentType
    severity: str  # "CRITICAL", "HIGH", "MEDIUM", "LOW"
    flight: Flight
    description: str
    impact: str
    timestamp: str
    status: str  # "OPEN", "IN_PROGRESS", "RESOLVED"
    owner: str
    actions_taken: List[str] = field(default_factory=list)
    # Store additional data as a dict field
    extra_data: Dict[str, Any] = field(default_factory=dict)  # Renamed from metadata

@dataclass
class Decision:
    incident_id: str
    recommendation: str
    actions: List[str]
    priority: Priority
    estimated_cost: float
    timeline: str
    risk_level: str
    confidence: float
    reasoning: str
    model_used: str
    timestamp: str

@dataclass
class AgentState:
    incidents: List[Incident] = field(default_factory=list)
    decisions: List[Decision] = field(default_factory=list)
    active_flights: List[Flight] = field(default_factory=list)
    last_update: str = datetime.now().isoformat()
    metrics: Dict[str, Any] = field(default_factory=dict)

# ============================================================================
# AOCC AGENT SYSTEM
# ============================================================================

class AOCCAgent:
    """Agentic AOCC System using GPT-5.6"""

    def __init__(self):
        self.client = client
        self.state = AgentState()

        # Model routing configuration
        self.model_routing = {
            "strategy": {
                "gpt-5.6-sol": ["critical", "complex", "analysis", "planning", "regulatory"],
                "gpt-5.6-terra": ["medium", "balanced", "general", "coordination"],
                "gpt-5.6-luna": ["simple", "quick", "status", "notification"]
            },
            "thresholds": {
                "complexity_score": 7,  # 1-10, above this goes to SOL
                "urgency_score": 8,      # 1-10, above this goes to SOL
                "time_critical": 5       # minutes, below this goes to LUNA
            }
        }

        self.decision_templates = {
            "flight_delay": {
                "prompt": """
You are an AOCC (Airline Operations Control Center) agent managing flight delays.
Analyze this situation and provide a comprehensive action plan:

FLIGHT: {flight_number}
DEPARTURE: {departure} → {arrival}
SCHEDULED: {departure_time}
CURRENT STATUS: {status}
DELAY: {delay} minutes
PASSENGERS: {passengers}
CREW STATUS: {crew_status}
WEATHER: {weather}
ATC UPDATE: {atc_update}

Provide:
1. ROOT CAUSE ANALYSIS
2. IMMEDIATE ACTIONS (0-30 min)
3. SHORT TERM ACTIONS (30-120 min)
4. LONG TERM ACTIONS (2-6 hours)
5. PASSENGER COMMUNICATION STRATEGY
6. CREW MANAGEMENT PLAN
7. COST IMPACT ASSESSMENT
8. REGULATORY COMPLIANCE CHECK
9. ESCALATION PLAN
10. SUCCESS METRICS
"""
            },
            "crew_unavailable": {
                "prompt": """
You are an AOCC agent managing crew availability issues.

FLIGHT: {flight_number}
CREW ROLE: {role}
ISSUE: {issue}
TIME TO DEPARTURE: {time_to_departure}
STANDBY CREW: {standby_available}

Provide:
1. CREW REPLACEMENT PLAN
2. REST COMPLIANCE CHECK
3. IMPACT ASSESSMENT
4. RECOVERY ACTIONS
5. COMMUNICATION PLAN
"""
            },
            "weather": {
                "prompt": """
You are an AOCC agent managing weather-related disruptions.

FLIGHT: {flight_number}
WEATHER CONDITION: {condition}
AFFECTED AIRPORTS: {airports}
FORECAST: {forecast}
VISIBILITY: {visibility}
WIND SPEED: {wind_speed}

Provide:
1. ROUTE ALTERNATIVES
2. TIMELINE ADJUSTMENTS
3. SAFETY ASSESSMENT
4. PASSENGER NOTIFICATION PLAN
5. FUEL PLANNING
"""
            }
        }

        print("🏢 AOCC Agentic System Initialized")
        print(f"📋 Models: {list(self.model_routing['strategy'].keys())}")
        print(f"📊 Decision Templates: {len(self.decision_templates)}")

    def route_to_model(self, incident: Incident) -> str:
        """Intelligently route incident to the best GPT-5.6 model"""

        # Calculate urgency score
        urgency_score = {
            "CRITICAL": 10,
            "HIGH": 8,
            "MEDIUM": 5,
            "LOW": 3
        }.get(incident.severity, 5)

        # Calculate complexity based on incident type
        complexity_map = {
            IncidentType.FLIGHT_DELAY: 7,
            IncidentType.CREW_UNAVAILABLE: 8,
            IncidentType.WEATHER: 6,
            IncidentType.MAINTENANCE: 9,
            IncidentType.PASSENGER_ISSUE: 4,
            IncidentType.REGULATORY: 9,
            IncidentType.ATC: 6,
            IncidentType.FUEL: 5
        }
        complexity = complexity_map.get(incident.type, 5)

        # Determine model based on complexity and urgency
        if complexity >= 7 or urgency_score >= 8:
            model = "gpt-5.6-sol"
            print(f"🧠 Routing to SOL (Complexity: {complexity}, Urgency: {urgency_score})")
        elif complexity >= 4 and urgency_score >= 4:
            model = "gpt-5.6-terra"
            print(f"⚖️ Routing to TERRA (Complexity: {complexity}, Urgency: {urgency_score})")
        else:
            model = "gpt-5.6-luna"
            print(f"⚡ Routing to LUNA (Complexity: {complexity}, Urgency: {urgency_score})")

        return model

    def analyze_incident(self, incident: Incident) -> Decision:
        """Analyze an incident and generate a decision using the appropriate model"""

        print(f"\n{'='*60}")
        print(f"🚨 ANALYZING INCIDENT: {incident.id}")
        print(f"   Type: {incident.type.value}")
        print(f"   Severity: {incident.severity}")
        print(f"   Flight: {incident.flight.flight_number}")
        print(f"{'='*60}")

        # Route to best model
        model = self.route_to_model(incident)

        # Build prompt based on incident type
        template = self.decision_templates.get(
            incident.type.value,
            self.decision_templates["flight_delay"]
        )

        # Prepare prompt data
        prompt_data = {
            "flight_number": incident.flight.flight_number,
            "departure": incident.flight.departure,
            "arrival": incident.flight.arrival,
            "departure_time": incident.flight.departure_time,
            "status": incident.flight.status,
            "passengers": incident.flight.passengers,
            "description": incident.description,
            "impact": incident.impact,
            "severity": incident.severity,
            **self._get_context_data(incident)
        }

        # Fill template
        prompt = template["prompt"].format(**prompt_data)

        # Generate decision using GPT-5.6
        print(f"\n🤔 Generating decision with {model}...")
        start_time = time.time()

        try:
            response = self.client.responses.create(
                model=model,
                input=prompt
            )

            latency = time.time() - start_time
            decision_text = response.output_text

            print(f"✅ Decision generated in {latency:.2f}s")

            # Parse decision
            decision = self._parse_decision(incident, decision_text, model, latency)

            # Store decision
            self.state.decisions.append(decision)

            return decision

        except Exception as e:
            print(f"❌ Error generating decision: {e}")
            return self._create_fallback_decision(incident, str(e))

    def _get_context_data(self, incident: Incident) -> Dict:
        """Get additional context data for the prompt"""
        context = {
            "crew_status": "Available",
            "weather": "Clear",
            "atc_update": "Normal operations",
            "delay": 0,
            "role": "Unknown",
            "issue": incident.description,
            "time_to_departure": "Unknown",
            "standby_available": "No",
            "condition": "Unknown",
            "airports": "Unknown",
            "forecast": "Unknown",
            "visibility": "Unknown",
            "wind_speed": "Unknown"
        }

        # Get extra data safely
        extra = incident.extra_data if hasattr(incident, 'extra_data') else {}

        # Add incident-specific context
        if incident.type == IncidentType.FLIGHT_DELAY:
            context["delay"] = extra.get("delay", 45)
            context["weather"] = extra.get("weather", "Unknown")
            context["atc_update"] = extra.get("atc", "Unknown")
            context["crew_status"] = extra.get("crew_status", "Available")

        elif incident.type == IncidentType.CREW_UNAVAILABLE:
            context["role"] = extra.get("role", "Pilot")
            context["time_to_departure"] = extra.get("time_to_departure", "2 hours")
            context["standby_available"] = extra.get("standby", "No")

        elif incident.type == IncidentType.WEATHER:
            context["condition"] = extra.get("condition", "Unknown")
            context["airports"] = extra.get("airports", "Unknown")
            context["forecast"] = extra.get("forecast", "Unknown")
            context["visibility"] = extra.get("visibility", "Unknown")
            context["wind_speed"] = extra.get("wind_speed", "Unknown")

        return context

    def _parse_decision(self, incident: Incident, response: str, model: str, latency: float) -> Decision:
        """Parse the decision from the response"""

        # Try to extract structured data from the response
        actions = []
        recommendation = ""
        reasoning = response

        # Simple parsing - look for action items
        lines = response.split('\n')
        for line in lines:
            if any(key in line.lower() for key in ['action', 'step', 'recommend']):
                if line.strip() and not line.startswith('#'):
                    cleaned = line.strip()
                    if cleaned.startswith('-') or cleaned.startswith('*') or cleaned[0].isdigit():
                        actions.append(cleaned)

        # If no actions found, try to extract from bullet points
        if not actions:
            for line in lines:
                if line.strip().startswith(('•', '-', '*', '1.', '2.', '3.')):
                    actions.append(line.strip())

        # If still no actions, use the first few sentences
        if not actions:
            sentences = response.split('.')
            if sentences:
                recommendation = sentences[0][:100]
                actions = [s.strip() for s in sentences[1:4] if len(s.strip()) > 10]

        # Determine priority
        priority_map = {
            "CRITICAL": Priority.CRITICAL,
            "HIGH": Priority.HIGH,
            "MEDIUM": Priority.MEDIUM,
            "LOW": Priority.LOW
        }
        priority = priority_map.get(incident.severity, Priority.MEDIUM)

        return Decision(
            incident_id=incident.id,
            recommendation=recommendation or "See detailed reasoning",
            actions=actions or ["Monitor situation", "Communicate with stakeholders", "Document decision"],
            priority=priority,
            estimated_cost=self._estimate_cost(incident),
            timeline=self._estimate_timeline(incident),
            risk_level=self._assess_risk(incident),
            confidence=0.85,
            reasoning=reasoning[:500],
            model_used=model,
            timestamp=datetime.now().isoformat()
        )

    def _estimate_cost(self, incident: Incident) -> float:
        """Estimate cost impact of incident"""
        base_costs = {
            "CRITICAL": 50000,
            "HIGH": 20000,
            "MEDIUM": 5000,
            "LOW": 1000
        }
        base = base_costs.get(incident.severity, 5000)

        # Add passenger cost
        passenger_cost = incident.flight.passengers * 150

        return base + passenger_cost + random.uniform(-5000, 5000)

    def _estimate_timeline(self, incident: Incident) -> str:
        """Estimate timeline for resolution"""
        timelines = {
            "CRITICAL": "30-60 minutes",
            "HIGH": "60-120 minutes",
            "MEDIUM": "2-4 hours",
            "LOW": "4-24 hours"
        }
        return timelines.get(incident.severity, "2-4 hours")

    def _assess_risk(self, incident: Incident) -> str:
        """Assess risk level"""
        risk_map = {
            "CRITICAL": "HIGH",
            "HIGH": "MEDIUM-HIGH",
            "MEDIUM": "MEDIUM",
            "LOW": "LOW"
        }
        return risk_map.get(incident.severity, "MEDIUM")

    def _create_fallback_decision(self, incident: Incident, error: str) -> Decision:
        """Create a fallback decision when API fails"""
        return Decision(
            incident_id=incident.id,
            recommendation="Manual intervention required",
            actions=["Escalate to human supervisor", "Review situation manually", "Document error"],
            priority=self._get_priority_from_severity(incident.severity),
            estimated_cost=10000,
            timeline="Immediate",
            risk_level="HIGH",
            confidence=0.3,
            reasoning=f"Fallback due to error: {error}",
            model_used="fallback",
            timestamp=datetime.now().isoformat()
        )

    def _get_priority_from_severity(self, severity: str) -> Priority:
        """Convert severity string to Priority enum"""
        return {
            "CRITICAL": Priority.CRITICAL,
            "HIGH": Priority.HIGH,
            "MEDIUM": Priority.MEDIUM,
            "LOW": Priority.LOW
        }.get(severity, Priority.MEDIUM)

    def process_incident(self, incident: Incident) -> Decision:
        """Main entry point for processing an incident"""
        return self.analyze_incident(incident)

    def get_decision_summary(self, decision: Decision) -> str:
        """Get a human-readable summary of a decision"""
        summary = f"""
═══════════════════════════════════════════════════════
📋 DECISION SUMMARY - Incident: {decision.incident_id}
═══════════════════════════════════════════════════════

🎯 RECOMMENDATION: {decision.recommendation}

📌 ACTIONS:
{self._format_actions(decision.actions)}

⚠️ PRIORITY: {decision.priority.name}
💰 ESTIMATED COST: ${decision.estimated_cost:,.2f}
⏱️ TIMELINE: {decision.timeline}
🎲 RISK LEVEL: {decision.risk_level}
📊 CONFIDENCE: {decision.confidence*100:.0f}%
🤖 MODEL USED: {decision.model_used}

💡 REASONING:
{decision.reasoning[:300]}...

═══════════════════════════════════════════════════════
"""
        return summary

    def _format_actions(self, actions: List[str]) -> str:
        """Format actions for display"""
        if not actions:
            return "  - No specific actions identified"

        formatted = []
        for i, action in enumerate(actions[:10], 1):
            formatted.append(f"  {i}. {action}")
        return '\n'.join(formatted)

    def get_metrics(self) -> Dict:
        """Get system metrics"""
        return {
            "total_incidents": len(self.state.incidents),
            "total_decisions": len(self.state.decisions),
            "active_flights": len(self.state.active_flights),
            "last_update": self.state.last_update,
            "models_used": list(set(d.model_used for d in self.state.decisions))
        }

# ============================================================================
# SIMULATED AOCC ENVIRONMENT
# ============================================================================

class AOCCSimulator:
    """Simulate AOCC incidents for testing"""

    @staticmethod
    def create_flight(flight_num: str, departure: str, arrival: str) -> Flight:
        """Create a test flight"""
        now = datetime.now()
        return Flight(
            flight_number=flight_num,
            departure=departure,
            arrival=arrival,
            departure_time=(now + timedelta(hours=2)).isoformat(),
            arrival_time=(now + timedelta(hours=5)).isoformat(),
            status="SCHEDULED",
            gate=random.choice(["A1", "B3", "C5", "D2", "E8"]),
            aircraft=random.choice(["B737", "A320", "B787", "A350"]),
            passengers=random.randint(100, 250),
            crew=[f"P{random.randint(100,999)}", f"C{random.randint(100,999)}", f"F{random.randint(100,999)}"]
        )

    @staticmethod
    def create_incident(flight: Flight, incident_type: IncidentType, severity: str = "MEDIUM") -> Incident:
        """Create a test incident"""
        incident_data = {
            IncidentType.FLIGHT_DELAY: {
                "description": f"Flight {flight.flight_number} delayed due to operational issues",
                "impact": f"{flight.passengers} passengers affected",
                "extra_data": {"delay": 45, "weather": "Clear", "atc": "Normal", "crew_status": "Available"}
            },
            IncidentType.CREW_UNAVAILABLE: {
                "description": f"Crew member unavailable for {flight.flight_number}",
                "impact": "Flight may be delayed or cancelled",
                "extra_data": {"role": "Pilot", "time_to_departure": "2 hours", "standby": "No"}
            },
            IncidentType.WEATHER: {
                "description": f"Severe weather conditions at {flight.arrival}",
                "impact": "Potential arrival delays",
                "extra_data": {"condition": "Thunderstorms", "airports": flight.arrival, "forecast": "Improving",
                           "visibility": "2 miles", "wind_speed": "25 knots"}
            },
            IncidentType.MAINTENANCE: {
                "description": f"Mechanical issue detected on {flight.aircraft}",
                "impact": "Aircraft requires inspection",
                "extra_data": {"issue": "Engine sensor", "estimated_time": "3 hours"}
            },
            IncidentType.PASSENGER_ISSUE: {
                "description": "Passenger medical emergency",
                "impact": "May require flight diversion",
                "extra_data": {"condition": "Medical", "severity": "High"}
            },
            IncidentType.REGULATORY: {
                "description": "Regulatory compliance issue detected",
                "impact": "Fine potential if not resolved",
                "extra_data": {"regulation": "FAA Part 121", "deadline": "24 hours"}
            },
            IncidentType.ATC: {
                "description": "ATC congestion at departure airport",
                "impact": "Departure slot delay",
                "extra_data": {"delay": "30 minutes", "reason": "Traffic volume"}
            },
            IncidentType.FUEL: {
                "description": "Fuel price surge affecting profitability",
                "impact": "Cost increase for route",
                "extra_data": {"price_increase": "15%", "impact": "High"}
            }
        }

        data = incident_data.get(incident_type, incident_data[IncidentType.FLIGHT_DELAY])

        return Incident(
            id=f"INC-{datetime.now().strftime('%Y%m%d')}-{random.randint(1000,9999)}",
            type=incident_type,
            severity=severity,
            flight=flight,
            description=data.get("description", "Generic incident"),
            impact=data.get("impact", "Impact unknown"),
            timestamp=datetime.now().isoformat(),
            status="OPEN",
            owner=f"Agent-{random.randint(1,5)}",
            actions_taken=[],
            extra_data=data.get("extra_data", {})  # Use extra_data instead of metadata
        )

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Run the AOCC Agentic System"""

    print("=" * 80)
    print("🏢 AOCC AGENTIC SYSTEM - GPT-5.6 POWERED")
    print("=" * 80)

    # Initialize agent
    agent = AOCCAgent()
    simulator = AOCCSimulator()

    # Create test flights
    print("\n📋 Creating test flights...")
    flights = [
        simulator.create_flight("AA123", "JFK", "LAX"),
        simulator.create_flight("UA456", "ORD", "SFO"),
        simulator.create_flight("DL789", "ATL", "MIA"),
        simulator.create_flight("SW888", "LAS", "DEN"),
        simulator.create_flight("EK202", "DXB", "LHR")
    ]

    for flight in flights:
        agent.state.active_flights.append(flight)
        print(f"  ✈️ {flight.flight_number}: {flight.departure} → {flight.arrival} ({flight.passengers} pax)")

    # Generate incidents
    print("\n🚨 Generating test incidents...")
    incidents = [
        simulator.create_incident(flights[0], IncidentType.FLIGHT_DELAY, "HIGH"),
        simulator.create_incident(flights[1], IncidentType.CREW_UNAVAILABLE, "CRITICAL"),
        simulator.create_incident(flights[2], IncidentType.WEATHER, "MEDIUM"),
        simulator.create_incident(flights[3], IncidentType.MAINTENANCE, "HIGH"),
        simulator.create_incident(flights[4], IncidentType.ATC, "MEDIUM"),
    ]

    for incident in incidents:
        agent.state.incidents.append(incident)
        print(f"  🔴 {incident.id}: {incident.type.value} - {incident.severity}")

    # Process incidents
    print("\n" + "=" * 80)
    print("🔄 PROCESSING INCIDENTS WITH GPT-5.6")
    print("=" * 80)

    # Process only first 2 incidents to save time (since SOL takes long)
    for incident in incidents[:2]:
        print(f"\n{'='*80}")
        print(f"📋 Processing Incident: {incident.id}")
        print(f"{'='*80}")

        decision = agent.process_incident(incident)

        # Display decision summary
        print(agent.get_decision_summary(decision))

        # Display metrics
        print("\n📊 System Metrics:")
        metrics = agent.get_metrics()
        for key, value in metrics.items():
            print(f"  {key}: {value}")

        print("\n" + "-" * 80)
        time.sleep(1)

    # Final summary
    print("\n" + "=" * 80)
    print("✅ AOCC AGENTIC SYSTEM EXECUTION COMPLETE")
    print("=" * 80)
    print(f"\n📊 Final Statistics:")
    print(f"  • Total Incidents Processed: {len(agent.state.incidents)}")
    print(f"  • Total Decisions Generated: {len(agent.state.decisions)}")
    print(f"  • Active Flights: {len(agent.state.active_flights)}")
    print(f"  • Models Used: {set(d.model_used for d in agent.state.decisions)}")

    print("\n📈 Model Usage Breakdown:")
    model_counts = {}
    for d in agent.state.decisions:
        model_counts[d.model_used] = model_counts.get(d.model_used, 0) + 1

    for model, count in model_counts.items():
        print(f"  • {model}: {count} decisions")

    # Save results
    with open("aocc_decisions.json", "w") as f:
        json.dump({
            "timestamp": datetime.now().isoformat(),
            "metrics": agent.get_metrics(),
            "decisions": [
                {
                    "incident_id": d.incident_id,
                    "recommendation": d.recommendation,
                    "actions": d.actions,
                    "priority": d.priority.name,
                    "estimated_cost": d.estimated_cost,
                    "timeline": d.timeline,
                    "risk_level": d.risk_level,
                    "confidence": d.confidence,
                    "model_used": d.model_used
                }
                for d in agent.state.decisions
            ]
        }, f, indent=2)

    print(f"\n💾 Results saved to: aocc_decisions.json")
    print("=" * 80)

# ============================================================================
# RUN THE SYSTEM
# ============================================================================

if __name__ == "__main__":
    main()

🏢 AOCC AGENTIC SYSTEM - GPT-5.6 POWERED
🏢 AOCC Agentic System Initialized
📋 Models: ['gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-5.6-luna']
📊 Decision Templates: 3

📋 Creating test flights...
  ✈️ AA123: JFK → LAX (112 pax)
  ✈️ UA456: ORD → SFO (180 pax)
  ✈️ DL789: ATL → MIA (111 pax)
  ✈️ SW888: LAS → DEN (249 pax)
  ✈️ EK202: DXB → LHR (108 pax)

🚨 Generating test incidents...
  🔴 INC-20260709-5617: flight_delay - HIGH
  🔴 INC-20260709-8666: crew_unavailable - CRITICAL
  🔴 INC-20260709-4917: weather - MEDIUM
  🔴 INC-20260709-2857: maintenance - HIGH
  🔴 INC-20260709-9979: atc - MEDIUM

🔄 PROCESSING INCIDENTS WITH GPT-5.6

📋 Processing Incident: INC-20260709-5617

🚨 ANALYZING INCIDENT: INC-20260709-5617
   Type: flight_delay
   Severity: HIGH
   Flight: AA123
🧠 Routing to SOL (Complexity: 7, Urgency: 8)

🤔 Generating decision with gpt-5.6-sol...
✅ Decision generated in 65.18s

═══════════════════════════════════════════════════════
📋 DECISION SUMMARY - Incident: INC-20260709-5617
═════════

## VERSION2

In [11]:
# ============================================================================
# AOCC AGENTIC SYSTEM - COLAB OPTIMIZED
# ============================================================================

from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
import json
import time
import logging
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any, Tuple
from dataclasses import dataclass, asdict, field
from enum import Enum
from functools import lru_cache
import random

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

# ============================================================================
# DATA CLASSES
# ============================================================================

class Priority(Enum):
    CRITICAL = 1
    HIGH = 2
    MEDIUM = 3
    LOW = 4

class IncidentType(Enum):
    FLIGHT_DELAY = "flight_delay"
    CREW_UNAVAILABLE = "crew_unavailable"
    WEATHER = "weather"
    MAINTENANCE = "maintenance"
    PASSENGER_ISSUE = "passenger_issue"
    REGULATORY = "regulatory"
    ATC = "atc"
    FUEL = "fuel"

@dataclass
class Flight:
    flight_number: str
    departure: str
    arrival: str
    departure_time: str
    arrival_time: str
    status: str
    gate: str
    aircraft: str
    passengers: int
    crew: List[str]

@dataclass
class Incident:
    id: str
    type: IncidentType
    severity: str
    flight: Flight
    description: str
    impact: str
    timestamp: str
    status: str
    owner: str
    actions_taken: List[str] = field(default_factory=list)
    extra_data: Dict[str, Any] = field(default_factory=dict)

@dataclass
class Decision:
    incident_id: str
    recommendation: str
    actions: List[str]
    priority: Priority
    estimated_cost: float
    timeline: str
    risk_level: str
    confidence: float
    reasoning: str
    model_used: str
    timestamp: str
    execution_time: float = 0.0

@dataclass
class AgentState:
    incidents: List[Incident] = field(default_factory=list)
    decisions: List[Decision] = field(default_factory=list)
    active_flights: List[Flight] = field(default_factory=list)
    last_update: str = datetime.now().isoformat()
    metrics: Dict[str, Any] = field(default_factory=dict)
    alerts: List[Dict] = field(default_factory=list)

# ============================================================================
# SIMPLE CACHE
# ============================================================================

class SimpleCache:
    """Simple cache with TTL"""

    def __init__(self, max_size: int = 100, ttl: int = 300):
        self.cache = {}
        self.max_size = max_size
        self.ttl = ttl
        self.hits = 0
        self.misses = 0

    def get(self, key: str) -> Optional[Any]:
        if key in self.cache:
            data, timestamp = self.cache[key]
            if time.time() - timestamp < self.ttl:
                self.hits += 1
                return data
            else:
                del self.cache[key]
        self.misses += 1
        return None

    def set(self, key: str, value: Any):
        if len(self.cache) >= self.max_size:
            oldest_key = next(iter(self.cache))
            del self.cache[oldest_key]
        self.cache[key] = (value, time.time())

    def get_stats(self) -> Dict:
        total = self.hits + self.misses
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": self.hits / total if total > 0 else 0,
            "cache_size": len(self.cache)
        }

# ============================================================================
# SIMPLE MONITOR
# ============================================================================

class SimpleMonitor:
    """Simple monitoring system"""

    def __init__(self):
        self.alerts = []
        self.metrics = {
            "total_processed": 0,
            "successful": 0,
            "failed": 0,
            "avg_confidence": 0.0,
            "avg_cost": 0.0,
            "avg_latency": 0.0
        }

    def check_health(self, incident: Incident, decision: Decision) -> List[Dict]:
        alerts = []

        if decision.confidence < 0.7:
            alerts.append({
                "type": "LOW_CONFIDENCE",
                "incident_id": incident.id,
                "message": f"Low confidence: {decision.confidence:.1%}",
                "severity": "WARNING",
                "timestamp": datetime.now().isoformat()
            })

        if decision.estimated_cost > 100000:
            alerts.append({
                "type": "HIGH_COST",
                "incident_id": incident.id,
                "message": f"High cost impact: ${decision.estimated_cost:,.2f}",
                "severity": "CRITICAL",
                "timestamp": datetime.now().isoformat()
            })

        if decision.execution_time > 60:
            alerts.append({
                "type": "HIGH_LATENCY",
                "incident_id": incident.id,
                "message": f"Slow processing: {decision.execution_time:.1f}s",
                "severity": "WARNING",
                "timestamp": datetime.now().isoformat()
            })

        self.alerts.extend(alerts)
        return alerts

    def update_metrics(self, decision: Decision):
        self.metrics["total_processed"] += 1
        if decision.confidence >= 0.7:
            self.metrics["successful"] += 1
        else:
            self.metrics["failed"] += 1

        n = self.metrics["total_processed"]
        self.metrics["avg_confidence"] = (self.metrics["avg_confidence"] * (n-1) + decision.confidence) / n
        self.metrics["avg_cost"] = (self.metrics["avg_cost"] * (n-1) + decision.estimated_cost) / n
        self.metrics["avg_latency"] = (self.metrics["avg_latency"] * (n-1) + decision.execution_time) / n

    def get_dashboard(self) -> Dict:
        success_rate = (
            self.metrics["successful"] / self.metrics["total_processed"]
            if self.metrics["total_processed"] > 0 else 0
        )

        return {
            "metrics": self.metrics,
            "success_rate": success_rate,
            "alert_count": len(self.alerts),
            "critical_alerts": sum(1 for a in self.alerts if a["severity"] == "CRITICAL"),
            "alerts": self.alerts[-5:],
            "status": "HEALTHY" if success_rate >= 0.95 else "DEGRADED"
        }

# ============================================================================
# AOCC AGENT SYSTEM - SIMPLIFIED
# ============================================================================

class AOCCAgent:
    """Simplified AOCC Agent System - Colab Compatible"""

    def __init__(self):
        self.client = client
        self.state = AgentState()
        self.cache = SimpleCache(max_size=50, ttl=300)
        self.monitor = SimpleMonitor()

        # Model routing
        self.model_routing = {
            "gpt-5.6-sol": ["critical", "complex", "analysis", "planning", "regulatory", "maintenance"],
            "gpt-5.6-terra": ["medium", "balanced", "general", "coordination", "weather"],
            "gpt-5.6-luna": ["simple", "quick", "status", "notification", "fuel"]
        }

        self.decision_templates = self._init_templates()

        logger.info("🏢 AOCC Agentic System Initialized")
        logger.info(f"📋 Models: {list(self.model_routing.keys())}")

    def _init_templates(self) -> Dict:
        return {
            "flight_delay": {
                "prompt": """
You are an AOCC agent managing flight delays.

FLIGHT: {flight_number}
DEPARTURE: {departure} → {arrival}
SCHEDULED: {departure_time}
DELAY: {delay} minutes
PASSENGERS: {passengers}
CREW STATUS: {crew_status}

Provide:
1. IMMEDIATE ACTIONS (0-30 min)
2. SHORT TERM ACTIONS (30-120 min)
3. PASSENGER COMMUNICATION PLAN
4. COST IMPACT ASSESSMENT
5. SUCCESS METRICS
"""
            },
            "crew_unavailable": {
                "prompt": """
You are an AOCC agent managing crew issues.

FLIGHT: {flight_number}
CREW ROLE: {role}
ISSUE: {issue}
TIME TO DEPARTURE: {time_to_departure}
STANDBY CREW: {standby_available}

Provide:
1. CREW REPLACEMENT PLAN
2. REST COMPLIANCE CHECK
3. IMPACT ASSESSMENT
4. RECOVERY ACTIONS
"""
            },
            "weather": {
                "prompt": """
You are an AOCC agent managing weather disruptions.

FLIGHT: {flight_number}
WEATHER: {condition}
AIRPORT: {airports}
FORECAST: {forecast}

Provide:
1. ROUTE ALTERNATIVES
2. TIMELINE ADJUSTMENTS
3. SAFETY ASSESSMENT
4. PASSENGER NOTIFICATION PLAN
"""
            }
        }

    def route_to_model(self, incident: Incident) -> str:
        urgency = {"CRITICAL": 10, "HIGH": 8, "MEDIUM": 5, "LOW": 3}.get(incident.severity, 5)
        complexity = {
            IncidentType.REGULATORY: 9,
            IncidentType.MAINTENANCE: 9,
            IncidentType.CREW_UNAVAILABLE: 8,
            IncidentType.FLIGHT_DELAY: 7,
            IncidentType.WEATHER: 6,
            IncidentType.ATC: 6,
            IncidentType.FUEL: 5,
            IncidentType.PASSENGER_ISSUE: 4
        }.get(incident.type, 5)

        if incident.type in [IncidentType.REGULATORY, IncidentType.MAINTENANCE]:
            model = "gpt-5.6-sol"
        elif incident.severity == "MEDIUM" and incident.type != IncidentType.CREW_UNAVAILABLE:
            model = "gpt-5.6-terra"
        elif complexity >= 7 or urgency >= 8:
            model = "gpt-5.6-sol"
        elif complexity >= 4 and urgency >= 4:
            model = "gpt-5.6-terra"
        else:
            model = "gpt-5.6-luna"

        logger.info(f"📊 Routing to {model} (Complexity: {complexity}, Urgency: {urgency})")
        return model

    def _get_context(self, incident: Incident) -> Dict:
        context = {
            "delay": incident.extra_data.get("delay", 45),
            "crew_status": incident.extra_data.get("crew_status", "Available"),
            "weather": incident.extra_data.get("weather", "Clear"),
            "atc": incident.extra_data.get("atc", "Normal"),
            "role": incident.extra_data.get("role", "Pilot"),
            "time_to_departure": incident.extra_data.get("time_to_departure", "2 hours"),
            "standby_available": incident.extra_data.get("standby", "No"),
            "condition": incident.extra_data.get("condition", "Unknown"),
            "airports": incident.extra_data.get("airports", "Unknown"),
            "forecast": incident.extra_data.get("forecast", "Unknown"),
            "issue": incident.description
        }
        return context

    def analyze_incident(self, incident: Incident) -> Decision:
        logger.info(f"🚨 Analyzing: {incident.id} - {incident.type.value}")

        model = self.route_to_model(incident)
        template = self.decision_templates.get(incident.type.value, self.decision_templates["flight_delay"])
        context = self._get_context(incident)

        prompt_data = {
            "flight_number": incident.flight.flight_number,
            "departure": incident.flight.departure,
            "arrival": incident.flight.arrival,
            "departure_time": incident.flight.departure_time,
            "passengers": incident.flight.passengers,
            "severity": incident.severity,
            **context
        }

        prompt = template["prompt"].format(**prompt_data)

        logger.info(f"🤔 Generating decision with {model}...")
        start_time = time.time()

        try:
            response = self.client.responses.create(
                model=model,
                input=prompt
            )

            execution_time = time.time() - start_time
            logger.info(f"✅ Decision in {execution_time:.2f}s")

            decision = self._parse_decision(incident, response.output_text, model, execution_time)
            self.state.decisions.append(decision)

            # Update metrics
            self.monitor.update_metrics(decision)
            alerts = self.monitor.check_health(incident, decision)
            self.state.alerts.extend(alerts)

            return decision

        except Exception as e:
            logger.error(f"❌ Error: {e}")
            return self._fallback_decision(incident, str(e))

    def _parse_decision(self, incident: Incident, response: str, model: str, execution_time: float) -> Decision:
        lines = response.split('\n')
        actions = []

        for line in lines:
            cleaned = line.strip()
            if cleaned and (cleaned.startswith(('•', '-', '*', '1.', '2.', '3.', '4.', '5.'))):
                actions.append(cleaned)

        if not actions:
            sentences = response.split('.')
            actions = [s.strip() for s in sentences[:3] if len(s.strip()) > 10]

        priority = {
            "CRITICAL": Priority.CRITICAL,
            "HIGH": Priority.HIGH,
            "MEDIUM": Priority.MEDIUM,
            "LOW": Priority.LOW
        }.get(incident.severity, Priority.MEDIUM)

        return Decision(
            incident_id=incident.id,
            recommendation=actions[0] if actions else "See reasoning",
            actions=actions[:10],
            priority=priority,
            estimated_cost=self._estimate_cost(incident),
            timeline=self._estimate_timeline(incident),
            risk_level=self._assess_risk(incident),
            confidence=0.85,
            reasoning=response[:500],
            model_used=model,
            timestamp=datetime.now().isoformat(),
            execution_time=execution_time
        )

    def _estimate_cost(self, incident: Incident) -> float:
        base = {"CRITICAL": 50000, "HIGH": 20000, "MEDIUM": 5000, "LOW": 1000}.get(incident.severity, 5000)
        return base + incident.flight.passengers * 150 + random.uniform(-5000, 5000)

    def _estimate_timeline(self, incident: Incident) -> str:
        return {"CRITICAL": "30-60 min", "HIGH": "60-120 min", "MEDIUM": "2-4 hours", "LOW": "4-24 hours"}.get(incident.severity, "2-4 hours")

    def _assess_risk(self, incident: Incident) -> str:
        return {"CRITICAL": "HIGH", "HIGH": "MEDIUM-HIGH", "MEDIUM": "MEDIUM", "LOW": "LOW"}.get(incident.severity, "MEDIUM")

    def _fallback_decision(self, incident: Incident, error: str) -> Decision:
        return Decision(
            incident_id=incident.id,
            recommendation="Manual intervention required",
            actions=["Escalate to supervisor", "Review manually", "Document error"],
            priority=Priority.HIGH,
            estimated_cost=10000,
            timeline="Immediate",
            risk_level="HIGH",
            confidence=0.3,
            reasoning=f"Fallback: {error[:200]}",
            model_used="fallback",
            timestamp=datetime.now().isoformat(),
            execution_time=0.0
        )

    def get_summary(self, decision: Decision) -> str:
        return f"""
═══════════════════════════════════════════════════════
📋 DECISION - {decision.incident_id}
═══════════════════════════════════════════════════════
🎯 Recommendation: {decision.recommendation[:100]}

📌 Actions:
{chr(10).join(f'  {i+1}. {a[:80]}' for i, a in enumerate(decision.actions[:5]))}

⚠️ Priority: {decision.priority.name}
💰 Cost: ${decision.estimated_cost:,.2f}
⏱️ Timeline: {decision.timeline}
📊 Confidence: {decision.confidence*100:.0f}%
🤖 Model: {decision.model_used}
⚡ Time: {decision.execution_time:.2f}s
═══════════════════════════════════════════════════════
"""

    def get_metrics(self) -> Dict:
        model_counts = {}
        for d in self.state.decisions:
            model_counts[d.model_used] = model_counts.get(d.model_used, 0) + 1

        return {
            "total_incidents": len(self.state.incidents),
            "total_decisions": len(self.state.decisions),
            "active_flights": len(self.state.active_flights),
            "model_usage": model_counts,
            "metrics": self.monitor.metrics,
            "status": self.monitor.get_dashboard()["status"],
            "alerts": len(self.state.alerts)
        }

# ============================================================================
# SIMULATOR
# ============================================================================

class Simulator:
    @staticmethod
    def create_flight(num: str, dep: str, arr: str) -> Flight:
        now = datetime.now()
        return Flight(
            flight_number=num,
            departure=dep,
            arrival=arr,
            departure_time=(now + timedelta(hours=random.randint(1, 3))).isoformat(),
            arrival_time=(now + timedelta(hours=random.randint(4, 7))).isoformat(),
            status=random.choice(["SCHEDULED", "BOARDING", "DELAYED"]),
            gate=random.choice(["A1", "B3", "C5", "D2", "E8"]),
            aircraft=random.choice(["B737", "A320", "B787"]),
            passengers=random.randint(100, 250),
            crew=[f"P{random.randint(100,999)}", f"C{random.randint(100,999)}"]
        )

    @staticmethod
    def create_incident(flight: Flight, incident_type: IncidentType, severity: str) -> Incident:
        data = {
            IncidentType.FLIGHT_DELAY: {
                "description": f"Flight {flight.flight_number} delayed",
                "impact": f"{flight.passengers} passengers affected",
                "extra": {"delay": random.randint(30, 120), "crew_status": "Available"}
            },
            IncidentType.CREW_UNAVAILABLE: {
                "description": f"Crew unavailable for {flight.flight_number}",
                "impact": "Flight may be delayed",
                "extra": {"role": "Pilot", "standby": "No", "time_to_departure": "2 hours"}
            },
            IncidentType.WEATHER: {
                "description": f"Weather at {flight.arrival}",
                "impact": "Potential delays",
                "extra": {"condition": "Thunderstorms", "forecast": "Improving", "airports": flight.arrival}
            },
            IncidentType.MAINTENANCE: {
                "description": f"Maintenance on {flight.aircraft}",
                "impact": "Aircraft requires inspection",
                "extra": {"issue": "Engine sensor", "estimated_time": "3 hours"}
            }
        }

        d = data.get(incident_type, data[IncidentType.FLIGHT_DELAY])
        return Incident(
            id=f"INC-{datetime.now().strftime('%Y%m%d')}-{random.randint(1000,9999)}",
            type=incident_type,
            severity=severity,
            flight=flight,
            description=d["description"],
            impact=d["impact"],
            timestamp=datetime.now().isoformat(),
            status="OPEN",
            owner=f"Agent-{random.randint(1,3)}",
            extra_data=d.get("extra", {})
        )

# ============================================================================
# MAIN
# ============================================================================

def main():
    """Run AOCC System"""

    print("=" * 80)
    print("🏢 AOCC AGENTIC SYSTEM - PRODUCTION")
    print("=" * 80)

    agent = AOCCAgent()
    simulator = Simulator()

    # Create flights
    print("\n📋 Creating flights...")
    flights = [
        simulator.create_flight("AA123", "JFK", "LAX"),
        simulator.create_flight("UA456", "ORD", "SFO"),
        simulator.create_flight("DL789", "ATL", "MIA"),
        simulator.create_flight("SW888", "LAS", "DEN"),
        simulator.create_flight("EK202", "DXB", "LHR")
    ]

    for f in flights:
        agent.state.active_flights.append(f)
        print(f"  ✈️ {f.flight_number}: {f.departure} → {f.arrival} ({f.passengers} pax)")

    # Create incidents
    print("\n🚨 Creating incidents...")
    incidents = [
        simulator.create_incident(flights[0], IncidentType.FLIGHT_DELAY, "HIGH"),
        simulator.create_incident(flights[1], IncidentType.CREW_UNAVAILABLE, "CRITICAL"),
        simulator.create_incident(flights[2], IncidentType.WEATHER, "MEDIUM"),
        simulator.create_incident(flights[3], IncidentType.MAINTENANCE, "HIGH"),
    ]

    for inc in incidents:
        agent.state.incidents.append(inc)
        print(f"  🔴 {inc.id}: {inc.type.value} - {inc.severity}")

    # Process incidents
    print("\n" + "=" * 80)
    print("🔄 PROCESSING INCIDENTS")
    print("=" * 80)

    for incident in incidents:
        print(f"\n📋 Processing: {incident.id}")
        decision = agent.analyze_incident(incident)
        print(agent.get_summary(decision))
        time.sleep(0.5)

    # Dashboard
    print("\n" + "=" * 80)
    print("📊 DASHBOARD")
    print("=" * 80)

    metrics = agent.get_metrics()
    print(f"Total Incidents: {metrics['total_incidents']}")
    print(f"Total Decisions: {metrics['total_decisions']}")
    print(f"Active Flights: {metrics['active_flights']}")
    print(f"System Status: {metrics['status']}")
    print(f"Alerts: {metrics['alerts']}")
    print("\nModel Usage:")
    for model, count in metrics['model_usage'].items():
        print(f"  • {model}: {count} decisions")

    # Save results
    with open("aocc_results.json", "w") as f:
        json.dump({
            "timestamp": datetime.now().isoformat(),
            "metrics": metrics,
            "decisions": [
                {
                    "incident_id": d.incident_id,
                    "recommendation": d.recommendation[:100],
                    "actions": d.actions[:5],
                    "cost": d.estimated_cost,
                    "confidence": d.confidence,
                    "model": d.model_used
                }
                for d in agent.state.decisions
            ]
        }, f, indent=2)

    print(f"\n💾 Results: aocc_results.json")
    print("=" * 80)
    print("✅ DONE")

# ============================================================================
# RUN
# ============================================================================

main()

🏢 AOCC AGENTIC SYSTEM - PRODUCTION

📋 Creating flights...
  ✈️ AA123: JFK → LAX (162 pax)
  ✈️ UA456: ORD → SFO (122 pax)
  ✈️ DL789: ATL → MIA (247 pax)
  ✈️ SW888: LAS → DEN (129 pax)
  ✈️ EK202: DXB → LHR (175 pax)

🚨 Creating incidents...
  🔴 INC-20260709-8995: flight_delay - HIGH
  🔴 INC-20260709-5300: crew_unavailable - CRITICAL
  🔴 INC-20260709-2398: weather - MEDIUM
  🔴 INC-20260709-9102: maintenance - HIGH

🔄 PROCESSING INCIDENTS

📋 Processing: INC-20260709-8995

═══════════════════════════════════════════════════════
📋 DECISION - INC-20260709-8995
═══════════════════════════════════════════════════════
🎯 Recommendation: **Scheduled departure:** July 10, 2026, 00:49 EDT

📌 Actions:
  1. **Scheduled departure:** July 10, 2026, 00:49 EDT
  2. **Current estimated departure:** **02:02 EDT**
  3. **Delay:** 73 minutes
  4. **Passengers:** 162
  5. **Crew:** Available

⚠️ Priority: HIGH
💰 Cost: $47,622.76
⏱️ Timeline: 60-120 min
📊 Confidence: 85%
🤖 Model: gpt-5.6-sol
⚡ Time: 35.35s


## 🔬 GPT-5.6 MAX REASONING MODE DEMO

In [14]:
# ============================================================================
# GPT-5.6 MAX REASONING DEMO - WORKING
# ============================================================================

from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
import time
from datetime import datetime

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

print("=" * 80)
print("🔬 GPT-5.6 MAX REASONING MODE DEMO")
print("=" * 80)
print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

# ============================================================================
# REASONING TEST PROMPTS
# ============================================================================

test_scenarios = [
    {
        "name": "🧠 Complex Operations Recovery",
        "prompt": """You are a senior aviation operations analyst. Create a recovery plan for this scenario:

- 3 aircraft grounded at different airports
- 15 connecting flights affected (120 passengers each)
- Weather deteriorating at hub (thunderstorms in 4 hours)
- 8 crew members nearing duty limits
- 2 alternate airports available (45-min diversion)
- Fuel prices up 12%
- Regulatory rest requirements apply

Provide:
1. Immediate actions (0-30 min)
2. Short-term plan (30-120 min)
3. Long-term strategy (2-6 hours)
4. Passenger communication plan
5. Cost impact estimate
6. Risk mitigation strategies
7. Success metrics"""
    },
    {
        "name": "📊 Fleet Renewal Analysis",
        "prompt": """You are a CFO analyzing fleet renewal:

Current fleet: 150 aircraft (avg age 12 years)
Fuel efficiency: 3.2 L/100km/passenger
New aircraft efficiency: 2.1 L/100km/passenger
Cost per new aircraft: $120M
Fuel price: $85/barrel (projected $95-120)
Maintenance costs: +8% annually for old fleet
Financing: 5% interest over 20 years
Resale value old fleet: $20M each
Passenger growth: 4% annually
Carbon tax expected: $50/ton CO2 in 3 years

Calculate:
1. Break-even point for replacement
2. 5-year financial impact
3. Environmental benefit
4. Recommended replacement schedule"""
    },
    {
        "name": "🏗️ System Architecture Design",
        "prompt": """Design a next-gen airline operations system:

Requirements:
- 50,000 concurrent users
- Real-time data (millions of events/sec)
- 99.999% uptime
- 200+ external system integrations
- Mobile-first for 15,000 field staff
- AI-powered predictive maintenance
- Real-time route optimization
- SOC2 Type II, GDPR, CCPA compliant
- Global deployment (12 regions)
- Disaster recovery: 4-hour RTO
- Budget: $50M over 3 years

Provide:
1. Architecture overview
2. Technology stack
3. Deployment strategy
4. Security architecture
5. Data pipeline design
6. Migration plan
7. Resource estimates
8. Risk assessment"""
    }
]

# ============================================================================
# MODELS TO TEST
# ============================================================================

models = [
    {"name": "🧠 MAX REASONING", "model": "gpt-5.6-sol", "desc": "Deep analysis"},
    {"name": "⚡ STANDARD", "model": "gpt-5.6-terra", "desc": "Balanced performance"},
    {"name": "🚀 FAST", "model": "gpt-5.6-luna", "desc": "Quick responses"}
]

# ============================================================================
# RUN DEMO
# ============================================================================

print("\n📋 Test Scenarios:")
for i, s in enumerate(test_scenarios, 1):
    print(f"   {i}. {s['name']}")

print("\n📊 Models:")
for i, m in enumerate(models, 1):
    print(f"   {i}. {m['name']} ({m['model']})")

print("\n" + "=" * 80)
print("🚀 RUNNING TESTS")
print("=" * 80)

all_results = []

# Run first scenario with all 3 models
test_scenario = test_scenarios[0]

print(f"\n📝 SCENARIO: {test_scenario['name']}")
print("=" * 80)

for model_info in models:
    model = model_info['model']
    model_name = model_info['name']

    print(f"\n🤖 Testing: {model_name} ({model})")
    print("-" * 60)

    try:
        start = time.time()

        # ✅ SIMPLE - No extra parameters
        response = client.responses.create(
            model=model,
            input=test_scenario['prompt']
        )

        elapsed = time.time() - start
        output = response.output_text

        print(f"✅ Success in {elapsed:.2f}s")
        print(f"📏 Length: {len(output):,} characters")

        # Show preview
        preview = output[:300] + "..." if len(output) > 300 else output
        print(f"\n📄 Response Preview:")
        print("-" * 40)
        print(preview)

        # Count sections
        sections = output.count('\n\n')
        bullets = output.count('•') + output.count('-') + output.count('*')

        print(f"\n📊 Stats:")
        print(f"   • Sections: {sections}")
        print(f"   • Bullet points: {bullets}")

        all_results.append({
            "model": model,
            "model_name": model_name,
            "time": elapsed,
            "length": len(output),
            "sections": sections,
            "bullets": bullets,
            "response": output
        })

    except Exception as e:
        print(f"❌ Error: {e}")

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("📊 COMPARISON SUMMARY")
print("=" * 80)

if all_results:
    print(f"\n{'Model':<20} {'Time':<12} {'Length':<12} {'Sections':<10}")
    print("-" * 60)

    for r in all_results:
        print(f"{r['model_name']:<20} {r['time']:>6.2f}s    {r['length']:>8,}    {r['sections']:>8}")

    # Recommendations
    fastest = min(all_results, key=lambda x: x['time'])
    longest = max(all_results, key=lambda x: x['length'])

    print("\n" + "-" * 60)
    print("💡 RECOMMENDATIONS:")
    print(f"   🏆 Best for complex analysis: {longest['model_name']} ({longest['length']:,} chars)")
    print(f"   ⚡ Best for speed: {fastest['model_name']} ({fastest['time']:.2f}s)")
    print(f"   📊 Average response time: {sum(r['time'] for r in all_results)/len(all_results):.2f}s")

    # Sample from best
    print("\n📄 SAMPLE FROM BEST RESPONSE:")
    print("-" * 60)
    best = max(all_results, key=lambda x: x['length'])
    print(best['response'][:500] + "...")

else:
    print("\n❌ No results. Please check your API key and model access.")

print("\n" + "=" * 80)
print("✅ DEMO COMPLETE")
print("=" * 80)

# ============================================================================
# OPTIONAL: RUN ALL SCENARIOS
# ============================================================================

print("\n" + "=" * 80)
print("🔄 RUNNING ALL SCENARIOS (Quick Mode)")
print("=" * 80)

for scenario in test_scenarios:
    print(f"\n📝 Scenario: {scenario['name']}")
    print("-" * 40)

    # Use LUNA for speed
    try:
        start = time.time()
        response = client.responses.create(
            model="gpt-5.6-luna",
            input=scenario['prompt']
        )
        elapsed = time.time() - start
        output = response.output_text

        print(f"✅ Generated in {elapsed:.2f}s")
        print(f"📏 Length: {len(output):,} chars")
        print(f"📄 Preview: {output[:150]}...")

    except Exception as e:
        print(f"❌ Error: {e}")

    time.sleep(0.5)

print("\n" + "=" * 80)
print("✅ ALL SCENARIOS COMPLETE")
print("=" * 80)

🔬 GPT-5.6 MAX REASONING MODE DEMO
Time: 2026-07-09 22:01:10

📋 Test Scenarios:
   1. 🧠 Complex Operations Recovery
   2. 📊 Fleet Renewal Analysis
   3. 🏗️ System Architecture Design

📊 Models:
   1. 🧠 MAX REASONING (gpt-5.6-sol)
   2. ⚡ STANDARD (gpt-5.6-terra)
   3. 🚀 FAST (gpt-5.6-luna)

🚀 RUNNING TESTS

📝 SCENARIO: 🧠 Complex Operations Recovery

🤖 Testing: 🧠 MAX REASONING (gpt-5.6-sol)
------------------------------------------------------------
✅ Success in 77.28s
📏 Length: 17,021 characters

📄 Response Preview:
----------------------------------------
## Recovery objectives

1. Protect safety, crew legality, and regulatory compliance.
2. Reduce exposure to the hub before thunderstorms arrive in approximately four hours.
3. Recover as much of the 15-flight/1,800-passenger schedule as possible.
4. Avoid unplanned diversions and passenger/crew stran...

📊 Stats:
   • Sections: 101
   • Bullet points: 394

🤖 Testing: ⚡ STANDARD (gpt-5.6-terra)
-----------------------------------------